In [0]:
import json
from pyspark.sql.types import *
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
# sprint schema
sprint_schema = StructType([
    StructField("id"       , IntegerType(), True),
    StructField("name"     , StringType() , True),
    StructField("state"    , StringType() , True),
    StructField("startDate", StringType() , True),
    StructField("endDate"  , StringType() , True),
    StructField("goal"     , StringType() , True),
    StructField("boardId"  , IntegerType(), True),
])

# sprint parser
def parse_sprint(sprint):
    return {"raw_json": json.dumps(sprint)}

# boards config — parent of sprints
BOARDS_CONFIG = {
    "name"          : "boards",
    "endpoint_key"  : "boards",
    "data_key"      : "values",
    "strategy"      : "flat",
    "result_fields" : ["id"],     # sprints need board id
    "schema"        : None,
    "parser"        : lambda x: {"raw_json": json.dumps(x)},
    "df_schema"     : StructType([StructField("raw_json", StringType(), True)]),
    "raw_table"     : None,
    "selected_table": None,
    "selected_columns": [],
    "raw_write_mode"     : "append",
    "selected_write_mode": "append",
    "log_write_mode"     : "append",
    "merge_key"          : None,
    "version_key": "agile_version",
}

# sprints config — child of boards
SPRINTS_CONFIG = {
    "name"          : "sprints",
    "version_key": "agile_version", 
    "endpoint_key"  : "sprints",
    "data_key"      : "values",
    "strategy"      : "per_parent",
    "parent": {
        "source"        : "boards",
        "field"         : "id",
        "param_name"    : "boardId",
        "param_template": "{VALUE}"
    },
    "schema"        : sprint_schema,
    "parser"        : parse_sprint,
    "raw_table"     : "sprints_raw",
    "selected_table": "sprints_selected",
    "selected_columns": [
        ("parsed.id"       , "id"        ),
        ("parsed.name"     , "name"      ),
        ("parsed.state"    , "state"     ),
        ("parsed.startDate", "start_date"),
        ("parsed.endDate"  , "end_date"  ),
    ],
    "raw_write_mode"     : "append",
    "selected_write_mode": "merge",
    "log_write_mode"     : "append",
    "merge_key"          : "id",
}